In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

try:
    # 1. Load the dataset
    train_df = pd.read_csv('data/train.tsv', sep='\t', header=None)
    test_df = pd.read_csv('data/test.tsv', sep='\t', header=None)
    
    # Print the unique labels to see exactly what is in the file
    print("Unique labels found in the training data:", train_df[1].unique())

    # Extract the text (index 2) and labels (index 1)
    X_train = train_df[2].fillna('') 
    y_train = train_df[1].astype(str) # Force as string to prevent errors
    X_test = test_df[2].fillna('')
    y_test = test_df[1].astype(str)

    # Convert multi-class labels to binary (Fake vs Real) safely
    valid_labels = ['true', 'mostly-true', 'half-true']
    
    # .lower() makes it lowercase, .strip() removes hidden spaces
    y_train_binary = y_train.apply(lambda x: 1 if x.lower().strip() in valid_labels else 0)
    y_test_binary = y_test.apply(lambda x: 1 if x.lower().strip() in valid_labels else 0)

    # 2. Vectorize the text using TF-IDF
    print("Vectorizing text...")
    vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # 3. Train the Logistic Regression Model
    print("Training model...")
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_tfidf, y_train_binary)

    # 4. Evaluate the model
    predictions = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test_binary, predictions)
    print(f"Model Test Accuracy: {accuracy * 100:.2f}%")

    # 5. Save the model and vectorizer for the Flask API
    print("Saving model and vectorizer...")
    joblib.dump(model, 'model.pkl')
    joblib.dump(vectorizer, 'vectorizer.pkl')
    print("Done! Files saved.")

except FileNotFoundError:
    print("Error: Could not find the dataset files. Make sure train2.tsv and test2.tsv are in the 'data' folder.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error: Could not find the dataset files. Make sure train2.tsv and test2.tsv are in the 'data' folder.
